In [ ]:
# --- paths come from human/config.py (auto-inserted by fix_notebooks.py) ---
import sys; sys.path.append('..')
from config import HUMAN_BASE


# Build TF-TF Protein-Protein Interaction Prior from STRING

**Goal**: Construct an 88 × 88 symmetric binary matrix of experimentally-supported protein-protein interactions, aligned to ChIP-DNA prior dimensions.

**Filtering strategy** (two-stage):
1. Query STRING with `required_score=400` (medium confidence) to retrieve all candidate interactions
2. Keep only edges with **experimental evidence**: `escore > 0` OR `dscore > 0`
   - `escore`: experimental (biochemical/biophysical assays)
   - `dscore`: curated databases (KEGG, Reactome, BioCarta etc.)
   - Excludes text-mining (`tscore`) which reflects literature co-mention, not physical interaction

**Rationale**: The original combined_score ≥ 900 filter retained 72 edges, but 100% were driven by `tscore` (text mining). This over-inflated complex sizes (max degree=18) because hub TFs like RUNX1 appear frequently in hematopoiesis literature alongside many partners. Restricting to experimental/database evidence yields smaller, biologically meaningful complexes.

**STRING version**: v12.0 stable URL for paper reproducibility.

## Part A: Setup and load gene list

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
from pathlib import Path
from io import StringIO

STRING_API_URL    = 'https://version-12-0.string-db.org/api'
SPECIES_TAXON     = 9606    # human
REQUIRED_SCORE    = 900     # high confidence combined score
MIN_ESCORE        = 0.6     # additionally require experimental evidence (escore > this)
CALLER_IDENTITY   = 'BMMC_SETIA_prior_builder'

GENE_LIST_PATH = f"{HUMAN_BASE}/bmmc_gene_list.tsv"
OUTPUT_DIR     = Path(f"{HUMAN_BASE}/GTEx_v11/ppi_prior_output");  OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR      = Path(f"{HUMAN_BASE}/GTEx_v11/ppi_prior_cache");   CACHE_DIR.mkdir(exist_ok=True)

print(f'STRING API: {STRING_API_URL}')
print(f'Species: {SPECIES_TAXON} (human)')
print(f'combined_score cutoff: >= {REQUIRED_SCORE}')
print(f'Additional filter: escore > {MIN_ESCORE} (experimental evidence required)')

In [ ]:
# Load gene list with SAME logic as ChIP prior — critical for matrix alignment
gene_df = pd.read_csv(GENE_LIST_PATH, sep='\t', comment='#')
all_genes    = gene_df['gene_symbol'].tolist()
tf_genes     = gene_df[gene_df['category'] != 'marker']['gene_symbol'].tolist()
marker_genes = gene_df[gene_df['category'] == 'marker']['gene_symbol'].tolist()
tf_set = set(tf_genes)

print(f'Total genes (matrix dim): {len(all_genes)}')
print(f'TFs (PPI source/target):  {len(tf_genes)}')
print(f'Markers (no PPI query):   {len(marker_genes)}')
print(f'\nFirst 10 TFs:   {tf_genes[:10]}')
print(f'First 5 markers: {marker_genes[:5]}')

# CRITICAL alignment: all_genes prefix MUST equal tf_genes (so rows 0..55 are TFs)
assert all_genes[:len(tf_genes)] == tf_genes, 'all_genes prefix must match tf_genes order!'
print('\n✓ Gene list ordering aligned (TFs are first 56 rows in all_genes)')

## Part B: Map TF gene symbols to STRING IDs

In [ ]:
MAPPING_CACHE = CACHE_DIR / 'string_id_mapping.tsv'

if MAPPING_CACHE.exists():
    print(f'Loading cached mapping from {MAPPING_CACHE}')
    mapping_df = pd.read_csv(MAPPING_CACHE, sep='\t')
else:
    print(f'Calling STRING get_string_ids for {len(tf_genes)} TFs...')
    params = {
        'identifiers'     : '\r'.join(tf_genes),
        'species'         : SPECIES_TAXON,
        'echo_query'      : 1,
        'caller_identity' : CALLER_IDENTITY,
    }
    response = requests.post(f'{STRING_API_URL}/tsv/get_string_ids', data=params)
    response.raise_for_status()
    print(f'Status: {response.status_code}, response size: {len(response.text):,} chars')
    mapping_df = pd.read_csv(StringIO(response.text), sep='\t')
    mapping_df.to_csv(MAPPING_CACHE, sep='\t', index=False)

print(f'\nMapping result: {mapping_df.shape}')
print(mapping_df.head())

In [ ]:
# Build gene_symbol -> (string_id, preferredName); keep FIRST hit per query (best match)
mapping_df.columns = [c.strip() for c in mapping_df.columns]
gene_to_string = {}
for _, row in mapping_df.iterrows():
    q   = row.get('queryItem')
    sid = row.get('stringId')
    pref = row.get('preferredName')
    if pd.notna(q) and pd.notna(sid) and q not in gene_to_string:
        gene_to_string[q] = (sid, pref)

missing_tfs = [tf for tf in tf_genes if tf not in gene_to_string]
mapped_tfs  = [tf for tf in tf_genes if tf in gene_to_string]
print(f'Mapped: {len(mapped_tfs)} / {len(tf_genes)} TFs')
if missing_tfs:
    print(f'MISSING (need manual review): {missing_tfs}')

# Show samples + flag any preferredName mismatch
print('\nSample mappings:')
for tf in tf_genes[:5]:
    if tf in gene_to_string:
        sid, pref = gene_to_string[tf]
        note = '' if pref == tf else f' (preferredName={pref})'
        print(f'  {tf:8s} -> {sid}{note}')

mismatches = [(tf, pref) for tf, (sid, pref) in gene_to_string.items() if pref != tf]
if mismatches:
    print(f'\nWARNING: {len(mismatches)} TFs have preferredName != query symbol — verify:')
    for tf, pref in mismatches:
        print(f'  query={tf} -> preferredName={pref}')
else:
    print('\n✓ All preferredNames match query symbols')

## Part C: Query STRING network for TF-TF interactions

In [ ]:
NETWORK_CACHE = CACHE_DIR / f'string_network_score{REQUIRED_SCORE}.tsv'

if NETWORK_CACHE.exists():
    print(f'Loading cached network from {NETWORK_CACHE}')
    network_df = pd.read_csv(NETWORK_CACHE, sep='\t')
else:
    print(f'Calling STRING network API for {len(mapped_tfs)} mapped TFs...')
    string_ids_to_query = [gene_to_string[tf][0] for tf in mapped_tfs]
    params = {
        'identifiers'     : '\r'.join(string_ids_to_query),
        'species'         : SPECIES_TAXON,
        'required_score'  : REQUIRED_SCORE,
        'caller_identity' : CALLER_IDENTITY,
    }
    response = requests.post(f'{STRING_API_URL}/tsv/network', data=params)
    response.raise_for_status()
    print(f'Status: {response.status_code}, response size: {len(response.text):,} chars')
    network_df = pd.read_csv(StringIO(response.text), sep='\t')
    network_df.to_csv(NETWORK_CACHE, sep='\t', index=False)

print(f'\nNetwork shape: {network_df.shape}')
print(f'Columns: {list(network_df.columns)}')
print(network_df.head())

print(f'\nTotal interactions: {len(network_df)}')
print(f'Score range: {network_df["score"].min():.3f} to {network_df["score"].max():.3f}')
for p in [25, 50, 75, 90]:
    print(f'  {p}th percentile: {np.percentile(network_df["score"], p):.3f}')
assert (network_df['score'] >= REQUIRED_SCORE/1000).all(), f'Some scores below {REQUIRED_SCORE/1000}!'
print(f'✓ All returned scores >= {REQUIRED_SCORE/1000}')

## Part D: Build 88 × 88 symmetric matrix

In [ ]:
# Reverse map: STRING preferredName -> our gene symbol
string_to_gene = {pref: tf for tf, (sid, pref) in gene_to_string.items()}

# ── Stage 2 filter: keep only experimental or database evidence ──
pre_filter = len(network_df)
network_exp = network_df[network_df['escore'] > MIN_ESCORE].copy()
print(f'Before filter: {pre_filter} edges (combined_score >= {REQUIRED_SCORE})')
print(f'After escore filter: {len(network_exp)} edges')
print(f'Removed (no experimental evidence): {pre_filter - len(network_exp)}')

# Score distribution of kept edges
if len(network_exp) > 0:
    print(f'\nKept edges score stats:')
    print(f'  escore: mean={network_exp["escore"].mean():.3f}  max={network_exp["escore"].max():.3f}')
    print(f'  dscore: mean={network_exp["dscore"].mean():.3f}  max={network_exp["dscore"].max():.3f}')

# Parse interactions, build symmetric edge set
tf_tf_edges = set()
skipped = []
for _, row in network_exp.iterrows():
    a = string_to_gene.get(row['preferredName_A'])
    b = string_to_gene.get(row['preferredName_B'])
    if a is None or b is None:
        skipped.append((row['preferredName_A'], row['preferredName_B']))
        continue
    tf_tf_edges.add((a, b))
    tf_tf_edges.add((b, a))

n_undirected = len(tf_tf_edges) // 2
print(f'\nUndirected TF-TF interactions (exp/db evidence): {n_undirected}')
print(f'Skipped (preferredName not in our 56): {len(skipped)}')
if skipped:
    print(f'  First 5: {skipped[:5]}')


In [ ]:
# Build 88 x 88 matrix in all_genes order
gene_to_idx = {g: i for i, g in enumerate(all_genes)}
ppi_matrix = np.zeros((len(all_genes), len(all_genes)), dtype=int)

for (a, b) in tf_tf_edges:
    if a in gene_to_idx and b in gene_to_idx:
        ppi_matrix[gene_to_idx[a], gene_to_idx[b]] = 1

n_tf = len(tf_genes)
print(f'Matrix shape: {ppi_matrix.shape}')
print(f'Total entries (symmetric, both ways): {ppi_matrix.sum()}')
print(f'Undirected edges: {ppi_matrix.sum() // 2}')
print(f'Diagonal sum (should be 0): {np.trace(ppi_matrix)}')

# Symmetry assertion
assert (ppi_matrix == ppi_matrix.T).all(), 'Matrix not symmetric!'
print('✓ Matrix is symmetric')

# Marker rows/cols must be all 0
marker_row_sum = ppi_matrix[n_tf:, :].sum()
marker_col_sum = ppi_matrix[:, n_tf:].sum()
print(f'\nMarker rows sum: {marker_row_sum}  (expected 0)')
print(f'Marker cols sum: {marker_col_sum}  (expected 0)')
assert marker_row_sum == 0 and marker_col_sum == 0, 'Markers should have no PPI!'

# Density
tf_block = ppi_matrix[:n_tf, :n_tf]
tf_density = tf_block.sum() / (n_tf * (n_tf - 1)) * 100
print(f'\nDensity of TF x TF block (excl diagonal): {tf_density:.1f}%')
print(f'Density of full 88 x 88 matrix:            {ppi_matrix.sum() / (len(all_genes)**2) * 100:.1f}%')

In [ ]:
# Per-TF degree distribution + hubs + isolated
tf_degrees = ppi_matrix[:n_tf, :].sum(axis=1)
tf_degree_df = pd.DataFrame({'TF': tf_genes, 'degree': tf_degrees}).sort_values('degree', ascending=False)

print('Degree distribution per TF:')
print(pd.Series(tf_degrees, index=tf_genes).describe())

print(f'\nTop 10 hubs:')
print(tf_degree_df.head(10).to_string(index=False))

isolated = tf_degree_df[tf_degree_df['degree'] == 0]['TF'].tolist()
print(f'\nIsolated TFs (no high-conf PPI to other panel TFs): {len(isolated)}')
print(f'  {isolated}')

## Part E: Sanity check well-known TF complexes

In [ ]:
sanity_pairs = [
    ('GATA1', 'TAL1',  'heptad: erythroid HSC core'),
    ('GATA1', 'GATA2', 'GATA paralog interaction'),
    ('GATA1', 'KLF1',  'erythroid pair'),
    ('RUNX1', 'TAL1',  'heptad'),
    ('RUNX1', 'GATA2', 'heptad'),
    ('RUNX1', 'FLI1',  'heptad'),
    ('TAL1',  'FLI1',  'heptad'),
    ('TAL1',  'GATA2', 'heptad'),
    ('PAX5',  'EBF1',  'B-cell commitment'),
    ('CEBPA', 'SPI1',  'myeloid lineage'),
    ('TBX21', 'EOMES', 'Th1/NK paralogs'),
    ('TCF7',  'LEF1',  'T-cell WNT paralogs'),
    ('IRF8',  'TCF4',  'pDC cofactor (E2-2)'),
]

found = 0
scorable = 0
for a, b, desc in sanity_pairs:
    if a not in gene_to_idx or b not in gene_to_idx:
        print(f'  [SKIP] {a}-{b}  (not in panel)')
        continue
    if a not in tf_set or b not in tf_set:
        print(f'  [SKIP] {a}-{b}  (one is marker)')
        continue
    scorable += 1
    has_edge = ((a, b) in tf_tf_edges) or ((b, a) in tf_tf_edges)
    marker = '✓' if has_edge else '✗'
    if has_edge: found += 1
    print(f'  [{marker}] {a:6s} - {b:6s}  ({desc})')

print(f'\nSanity check: {found}/{scorable} known TF-TF interactions in prior')

## Part F: Save outputs

In [ ]:
# Matrix (88x88, tab-separated, no header)
matrix_path = OUTPUT_DIR / 'BMMC_TF_TF_PPI_matrix.txt'
np.savetxt(matrix_path, ppi_matrix, fmt='%d', delimiter='\t')
print(f'Saved: {matrix_path.name}  shape={ppi_matrix.shape}')

# Row/col names (identical because square + symmetric)
names = '\n'.join(all_genes)
(OUTPUT_DIR / 'BMMC_TF_TF_PPI_matrix_rownames.txt').write_text(names)
(OUTPUT_DIR / 'BMMC_TF_TF_PPI_matrix_colnames.txt').write_text(names)
print(f'Saved: BMMC_TF_TF_PPI_matrix_rownames.txt  ({len(all_genes)} genes)')
print(f'Saved: BMMC_TF_TF_PPI_matrix_colnames.txt  ({len(all_genes)} genes, same as rownames)')

# Raw STRING network with all partial scores
# Save both raw and filtered networks
raw_path = OUTPUT_DIR / 'BMMC_TF_TF_PPI_raw_network.tsv'
network_df.to_csv(raw_path, sep='\t', index=False)
print(f'Saved: {raw_path.name}  ({len(network_df)} STRING records, unfiltered)')

exp_path = OUTPUT_DIR / 'BMMC_TF_TF_PPI_exp_network.tsv'
network_exp.to_csv(exp_path, sep='\t', index=False)
print(f'Saved: {exp_path.name}  ({len(network_exp)} experimental/database records)')

# Per-TF degree summary
tf_degree_df.to_csv(OUTPUT_DIR / 'BMMC_TF_TF_PPI_per_tf_degrees.tsv', sep='\t', index=False)
print(f'Saved: BMMC_TF_TF_PPI_per_tf_degrees.tsv')

In [ ]:
# Yeast-format JSON (Cytoscape.cyjs)
# Matches ChIP prior JSON format exactly:
#   node: {id, label, is_tf}
#   edge: {source, target, label, style}
# For symmetric PPI, both (A->B) and (B->A) are stored, so yeast pipeline's
# get_edges_from_json() can do symmetric source-keyed lookup.

def ppi_edges_to_yeast_json(edge_set, all_node_names, tf_set, path):
    nodes = [
        {'id': n, 'label': n, 'is_tf': 1 if n in tf_set else 0}
        for n in all_node_names
    ]
    name_to_idx = {n: i for i, n in enumerate(all_node_names)}
    adj = {}
    for (a, b) in edge_set:
        adj.setdefault(a, set()).add(b)
    edges = []
    for src in all_node_names:
        if src not in adj:
            continue
        for tgt in sorted(adj[src], key=lambda x: name_to_idx.get(x, 1e9)):
            edges.append({
                'source': src,
                'target': tgt,
                'label' : '',
                'style' : ['solid', 'triangle'],
            })
    with open(path, 'w') as f:
        json.dump({'nodes': nodes, 'edges': edges}, f, indent=2)
    return len(nodes), len(edges)

json_path = OUTPUT_DIR / 'BMMC_TF_TF_PPI_prior.json'
n_n, n_e = ppi_edges_to_yeast_json(tf_tf_edges, all_genes, tf_set, json_path)
print(f'Saved: {json_path.name}')
print(f'  Nodes:  {n_n}')
print(f'  Edges:  {n_e} directed  ({n_e // 2} undirected)')

# Read-back verification
with open(json_path) as f:
    data = json.load(f)

print(f'\nRead-back verification:')
print(f'  Top-level keys: {list(data.keys())}')
n_tf_nodes     = sum(1 for n in data['nodes'] if n['is_tf'] == 1)
n_marker_nodes = sum(1 for n in data['nodes'] if n['is_tf'] == 0)
print(f'  TF nodes:     {n_tf_nodes}  (expected 56)')
print(f'  Marker nodes: {n_marker_nodes}  (expected 32)')
print(f'\n  Sample TF node:     {data["nodes"][0]}')
print(f'  Sample marker node: {data["nodes"][n_tf] if len(data["nodes"]) > n_tf else "N/A"}')
if data['edges']:
    print(f'  Sample edge:        {data["edges"][0]}')

In [ ]:
# Methods summary for paper
print('=' * 60)
print('Methods summary:')
print('=' * 60)
print(f"""
TF-TF protein-protein interaction prior was retrieved from STRING v12.0
(https://version-12-0.string-db.org/) via the REST API. The 56 hematopoietic
TFs in our panel were mapped to STRING identifiers using the get_string_ids
endpoint (species 9606). The network endpoint (/api/tsv/network) was then
queried with all 56 STRING IDs with combined_score >= 900 (highest confidence).
From the returned interactions, we additionally required experimental evidence
(escore > 0), retaining only interactions validated by biochemical or biophysical
assays. This excludes interactions supported solely by text mining (tscore), which
reflects literature co-mention rather than physical protein interaction, and prevents
artificially large complexes from inflating the GRN search space. The resulting interactions were assembled into an
88 x 88 symmetric binary matrix where rows and columns follow the same gene
order as the expression matrix and the ChIP-DNA prior (56 TFs followed by
32 markers); marker rows/columns are all zero by construction since they
have no associated TF activity. The resulting matrix contains
{ppi_matrix.sum() // 2} undirected high-confidence TF-TF interactions
(density {tf_density:.1f}% within the 56 x 56 TF block, excluding the diagonal).
""")